<a href="https://colab.research.google.com/github/ascordero001-cell/enares-2024-crs04-ml/blob/main/notebooks/01_ingesta/01_ENARES_2024_PROJECT_crear_estructura_drive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ENARES 2024 CRS04 ML Pipeline

## Stage 1 — Project Structure Initialization

### Notebook Information

| Field | Value |
|---|---|
| **Notebook name** | `01_ENARES_2024_PROJECT_crear_estructura_drive.ipynb` |
| **Role** | Computer Science Lead — Data Engineering |
| **Author** | Ana Cordero Ricaldi |
| **Account used** | `anacordero.001@gmail.com` |
| **Environment** | Google Colab + Google Drive |
| **Project scope** | ENARES 2024 CRS04 ML Pipeline |
| **Script version** | `stage1-drive-structure-v1.0` |
| **Date executed** | `[AUTO-GENERATED AT RUNTIME]` |

---

# OBJECTIVE

Create the official Google Drive folder structure required for the ENARES 2024 CRS04 ML Pipeline project.

This notebook initializes the project workspace, validates folder creation, and records the generated Google Drive folder IDs for reproducibility and auditing purposes.

---

# FUNCTION OF THIS NOTEBOOK

This notebook is responsible only for:

- authenticating Google Drive access;
- creating the standard project directory structure;
- validating whether folders already exist;
- preventing accidental duplicate folder creation;
- generating a registry of folder IDs;
- documenting the storage hierarchy used by the project.

---

# METHODOLOGICAL DECISION

Google Drive is used as the primary raw-data storage layer because:

- ENARES source files are large and version-sensitive;
- raw files must remain outside GitHub;
- reproducibility requires persistent storage paths;
- folder IDs allow traceable data lineage;
- project outputs can be centrally organised and audited.

The folder structure is standardised to ensure all future stages of the pipeline use consistent paths and storage conventions.

---

# STRICT LIMITS OF THIS NOTEBOOK

This notebook strictly **DOES NOT** perform:

- ingestion of ENARES files;
- downloading from INEI;
- extraction of ZIP packages;
- SHA-256 checksum generation;
- ETL processing;
- cleaning or transformation;
- BigQuery loading;
- statistical analysis;
- ML modelling.

Its sole responsibility is project infrastructure setup.

---

# EXPECTED GOOGLE DRIVE STRUCTURE

```text
ENARES_2024_CRS04_ML/
│
├── 01BasesDatosPrimarias/
│   ├── originales_zip/
│   ├── extraidos/
│   └── manifests/
│
├── 02BasesIntermedias/
│
├── 03BasesProcesadas/
│
├── 04CuestionariosInformes/
│   ├── cuestionarios/
│   ├── diccionarios/
│   └── reportes/
│
├── 05Resultados/
│   ├── logs/
│   ├── validaciones/
│   └── exports/
│
├── 06Scripts/
│
└── 07DocumentacionTecnica/
```

---

# EXPECTED OUTPUTS

```text
05Resultados/logs/
└── ENARES_2024_PROJECT_drive_folder_ids.csv
```

---

# REPRODUCIBILITY REQUIREMENT

The notebook must be executable from zero and must consistently reproduce:

- the same folder hierarchy;
- the same naming conventions;
- the same project structure logic;
- a verifiable folder ID registry.

All folder creation actions must be logged and auditable.

---

# CURRENT STATUS

Notebook prepared for:

- Google Drive authentication;
- root project folder creation;
- subfolder validation;
- folder ID registration;
- infrastructure verification for Stage 1 ingestion.

In [1]:
import os
import pandas as pd
from datetime import datetime
from google.colab import drive

In [2]:
from google.colab import auth
from googleapiclient.discovery import build
import google.auth

In [3]:
print("Authenticating with Google Drive API...")
auth.authenticate_user()
creds, _ = google.auth.default()
drive_service = build('drive', 'v3', credentials=creds)

Authenticating with Google Drive API...


In [4]:
from google.colab import drive
drive.mount('/content/drive')

ROOT_LOCAL_PATH = "/content/drive/MyDrive/ENARES_2024_PROJECT"
PROJECT_ROOT_NAME = "ENARES_2024_PROJECT"

folders = [
    "01BasesDatosPrimarias",
    "02BasesDatosSecundarias",
    "03BasesDatosAnaliticas",
    "04CuestionariosInformes",
    "05Resultados",
    "99Codigos",
    "99Codigos/01_ingesta",
    "99Codigos/02_bigquery",
    "99Codigos/03_limpieza",
    "99Codigos/04_validacion",
    "99Codigos/05_modelamiento",
    "04CuestionariosInformes/cuestionarios",
    "04CuestionariosInformes/fichas_tecnicas",
    "04CuestionariosInformes/diccionarios",
    "04CuestionariosInformes/reportes",
    "05Resultados/logs",
    "05Resultados/tablas",
    "05Resultados/graficos",
    "05Resultados/reportes"
]

# Helper function to get or create a folder via API
def get_or_create_folder(folder_name, parent_id):
    # Query to check if folder already exists in the specific parent folder
    query = f"name='{folder_name}' and '{parent_id}' in parents and mimeType='application/vnd.google-apps.folder' and trashed=false"
    results = drive_service.files().list(q=query, spaces='drive', fields='files(id, name)').execute()
    items = results.get('files', [])

    if not items:
        # Create it if it doesn't exist
        file_metadata = {
            'name': folder_name,
            'parents': [parent_id],
            'mimeType': 'application/vnd.google-apps.folder'
        }
        folder = drive_service.files().create(body=file_metadata, fields='id').execute()
        return folder.get('id'), "created"
    else:
        # Return the first match if it exists
        return items[0].get('id'), "verified"

# 3. Create/Verify Project Root
print(f"Locating/Creating root folder: {PROJECT_ROOT_NAME}...")
root_id, root_status = get_or_create_folder(PROJECT_ROOT_NAME, 'root')

# Dictionary to keep track of folder paths and their actual Drive IDs
folder_mapping = {"": root_id}
results = []
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# 4. Iterate through required folders and create them via API
print("Processing subfolders...")
for path in folders:
    parts = path.split('/')
    folder_name = parts[-1]
    parent_path = '/'.join(parts[:-1]) # Yields "" if it's a top-level folder

    # Look up the parent ID we saved earlier
    parent_id = folder_mapping[parent_path]

    # Get or create the current folder
    folder_id, status = get_or_create_folder(folder_name, parent_id)

    # Save the new ID into our mapping for any future child folders
    folder_mapping[path] = folder_id

    # Append to results matching Stage 1 expected columns
    results.append({
        "folder_name": path,
        "folder_id": folder_id,
        "parent_folder_id": parent_id,
        "status": status,
        "created_or_verified_date": timestamp
    })

    print(f"{status.upper():<10} | {path}")

# 5. Save the output to CSV
df = pd.DataFrame(results)

# Ensure the local directory exists for the CSV save (using traditional OS module)
output_dir = os.path.join(ROOT_LOCAL_PATH, "05Resultados/logs")
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "ENARES_2024_PROJECT_drive_folder_ids.csv")
df.to_csv(output_path, index=False)

print("\nSaved ID registry to:", output_path)
print("\nAll folders successfully processed via Drive API.")

Mounted at /content/drive
Locating/Creating root folder: ENARES_2024_PROJECT...
Processing subfolders...
VERIFIED   | 01BasesDatosPrimarias
VERIFIED   | 02BasesDatosSecundarias
VERIFIED   | 03BasesDatosAnaliticas
VERIFIED   | 04CuestionariosInformes
VERIFIED   | 05Resultados
VERIFIED   | 99Codigos
VERIFIED   | 99Codigos/01_ingesta
VERIFIED   | 99Codigos/02_bigquery
VERIFIED   | 99Codigos/03_limpieza
VERIFIED   | 99Codigos/04_validacion
VERIFIED   | 99Codigos/05_modelamiento
VERIFIED   | 04CuestionariosInformes/cuestionarios
VERIFIED   | 04CuestionariosInformes/fichas_tecnicas
VERIFIED   | 04CuestionariosInformes/diccionarios
VERIFIED   | 04CuestionariosInformes/reportes
VERIFIED   | 05Resultados/logs
VERIFIED   | 05Resultados/tablas
VERIFIED   | 05Resultados/graficos
VERIFIED   | 05Resultados/reportes

Saved ID registry to: /content/drive/MyDrive/ENARES_2024_PROJECT/05Resultados/logs/ENARES_2024_PROJECT_drive_folder_ids.csv

All folders successfully processed via Drive API.
